# Step 1.1 — NAIP Tile Export & Data Prep

Exports raw NAIP pixels as a grid of GeoTIFFs to GCS, downloads locally, and stitches into a GDAL VRT.

**Output:** `outputs/segmentation/nyc_segmentation_2022/naip_tiles/`
- `tile_*.tif` — individual NAIP tiles (R, G, B, NIR)
- `tasks.json` — GEE task IDs for resumability
- `naip_2022_nyc.vrt` — GDAL virtual raster
- `naip_2022_nyc.json` — sidecar: band order, CRS, bounds, tile count

## Cell 1 — Imports & Config

In [ ]:
from pathlib import Path

from vacant_lot.config import load_config
from vacant_lot.gee_utils import init_gee, load_image_collection
from vacant_lot.geometry import get_city_geometry
from vacant_lot.tile_export import (
    build_tile_grid,
    download_tiles_from_gcs,
    export_naip_tiles,
    merge_tiles_to_vrt,
    wait_for_exports,
)

REPO_ROOT = Path("..")  # data_prep/ is one level below repo root
CONFIG_FILE = "nyc_segmentation.yaml"

config = load_config(CONFIG_FILE, config_dir=REPO_ROOT / "config")
init_gee(config.gcp)

print(f"City: {config.city_name}")
print(f"Run key: {config._run_key()}")
print(f"NAIP year: {config.naip.year}")
print(f"Tile size: {config.segmentation.tile_size_deg}°")

## Cell 2 — Load NAIP Mosaic

In [ ]:
import ee

region = get_city_geometry(config)

naip_mosaic = load_image_collection(
    collection_id=config.naip.collection_id,
    start_date=config.naip.start_date,
    end_date=config.naip.end_date,
    region=region,
    mosaic=True,
)

# Export only raw R, G, B, NIR bands (spectral indices computed at dataset load time)
imagery = naip_mosaic.select(["R", "G", "B", "N"])

print("NAIP bands:", imagery.bandNames().getInfo())

## Cell 3 — Create Output Directories

In [ ]:
config.ensure_seg_output_dirs(REPO_ROOT)
tiles_dir = config.get_naip_tiles_dir(REPO_ROOT)
print(f"Tiles dir: {tiles_dir}")

## Cell 4 — Preview Tile Grid

In [ ]:
tiles = build_tile_grid(region, config.segmentation.tile_size_deg)
print(f"Total tiles: {len(tiles)}")
print(f"First tile: {tiles[0]}")
print(f"Last tile:  {tiles[-1]}")

## Cell 5 — Submit GEE Export Tasks

> **Note:** This submits ~100–200 export tasks. Check progress at
> https://code.earthengine.google.com/ → Tasks tab.
> Tasks persist in `tasks.json` for resumability.

In [ ]:
task_records = export_naip_tiles(
    config=config,
    region=region,
    imagery=imagery,
    scale=1,
    crs=config.raster.output_crs,
)

task_ids = [r["task_id"] for r in task_records]
gcs_paths = [r["gcs_path"] for r in task_records]

print(f"Submitted {len(task_records)} export tasks")
print(f"Example GCS path: {gcs_paths[0]}")

## Cell 6 — Wait for Exports (blocking)

> **Note:** This blocks until all tasks complete. NYC at 1m resolution may take
> several hours. Safe to run overnight or in a terminal with `nohup`.
> 
> To resume after a notebook restart, load `tasks.json` instead of resubmitting:
> ```python
> import json
> task_records = json.loads((tiles_dir / "tasks.json").read_text())
> task_ids = [r["task_id"] for r in task_records]
> gcs_paths = [r["gcs_path"] for r in task_records]
> ```

In [ ]:
wait_for_exports(task_ids, poll_interval=60, timeout_hours=8.0)

## Cell 7 — Download Tiles from GCS

In [ ]:
local_tile_paths = download_tiles_from_gcs(
    config=config,
    gcs_paths=gcs_paths,
    local_dir=tiles_dir,
)

print(f"Downloaded {len(local_tile_paths)} tiles to {tiles_dir}")

## Cell 8 — Build GDAL VRT

In [ ]:
output_vrt = tiles_dir / "naip_2022_nyc.vrt"

vrt_path = merge_tiles_to_vrt(
    tile_paths=local_tile_paths,
    output_vrt=output_vrt,
)

print(f"VRT created: {vrt_path}")

## Cell 9 — Verification

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt

with rasterio.open(vrt_path) as ds:
    print(f"VRT shape:  {ds.height} × {ds.width} px")
    print(f"Band count: {ds.count}  (expected 4: R, G, B, NIR)")
    print(f"CRS:        {ds.crs}")
    print(f"Bounds:     {ds.bounds}")

    assert ds.count == 4, f"Expected 4 bands, got {ds.count}"

    # Read a 256×256 window from the centre
    cx, cy = ds.width // 2, ds.height // 2
    window = Window(cx - 128, cy - 128, 256, 256)
    rgb = ds.read([1, 2, 3], window=window)  # R, G, B

assert rgb.max() > 0, "Window is all-zero — check tile coverage"
assert rgb.min() >= 0 and rgb.max() <= 255, f"Unexpected value range: [{rgb.min()}, {rgb.max()}]"

# Show RGB thumbnail
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(np.moveaxis(rgb, 0, -1))
ax.set_title("Centre window — NAIP RGB (256×256 px)")
ax.axis("off")
plt.tight_layout()
plt.show()

print("Verification passed.")